#<center>**TALLER 2 - SISTEMA RAG PARA ECOMARKET**</center>

En este taller implementamos un sistema de Retrieval Augmented Generation (RAG)
para mejorar la atención al cliente de EcoMarket.

Se  utiliza:

- LangChain para integrar los componentes.
- Sentence Transformers para generar embeddings.
- ChromaDB como base de datos vectorial.
- Un modelo de lenguaje para generar las respuestas.

Su flujo general es: Documentos → Chunking → Embeddings → ChromaDB → Retriever → LLM → Respuesta



In [2]:
!pip install -q \
langchain \
langchain-community \
langchain-chroma \
langchain-huggingface \
langchain-text-splitters \
chromadb \
sentence-transformers \
transformers \
accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 

## 1. Importación de librerías

En esta sección se importan las herramientas necesarias para cargar los documentos,
segmentarlos, generar embeddings, almacenar los vectores y construir el flujo RAG.

In [3]:
import os
import json
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

## 2. Cargar la base de conocimiento

Los documentos de EcoMarket se cargarán desde los archivos creados previamente:

- política de devoluciones;
- preguntas frecuentes;
- catálogo de productos;
- información de envíos.

In [5]:
documentos = []


# Política de devoluciones

with open("/content/politica_devoluciones.md", "r", encoding="utf-8") as archivo:
    texto = archivo.read()

documentos.append(
    Document(
        page_content=texto,
        metadata={
            "fuente": "politica_devoluciones.md",
            "tipo": "politica_devoluciones"
        }
    )
)

# Información de envíos

with open("/content/envios.md", "r", encoding="utf-8") as archivo:
    texto = archivo.read()

documentos.append(
    Document(
        page_content=texto,
        metadata={
            "fuente": "envios.md",
            "tipo": "envios"
        }
    )
)


# Preguntas frecuentes

with open("/content/faq.json", "r", encoding="utf-8") as archivo:
    faq = json.load(archivo)

for item in faq:
    contenido = (
        f"Pregunta: {item['pregunta']}\n"
        f"Respuesta: {item['respuesta']}"
    )

    documentos.append(
        Document(
            page_content=contenido,
            metadata={
                "fuente": "faq.json",
                "tipo": "pregunta_frecuente"
            }
        )
    )

# Catálogo de productos

productos = pd.read_csv("/content/productos.csv")

for _, producto in productos.iterrows():

    contenido = f"""
Código del producto: {producto['codigo_producto']}
Nombre del producto: {producto['nombre_producto']}
Categoría: {producto['categoria']}
Disponibilidad: {producto['disponibilidad']}
Admite devolución: {producto['admite_devolucion']}
Condición de devolución: {producto['condicion_devolucion']}
"""

    documentos.append(
        Document(
            page_content=contenido,
            metadata={
                "fuente": "productos.csv",
                "tipo": "producto",
                "codigo_producto": producto["codigo_producto"]
            }
        )
    )

print(f"Documentos cargados: {len(documentos)}")

Documentos cargados: 19


In [6]:
documentos[:3]

[Document(metadata={'fuente': 'politica_devoluciones.md', 'tipo': 'politica_devoluciones'}, page_content='# Política de Devoluciones de EcoMarket\n\nEcoMarket permite realizar devoluciones dentro de los 30 días posteriores a la fecha de compra.\n\nPara que un producto sea apto para devolución debe cumplir las siguientes condiciones:\n\n- El producto debe estar sin usar.\n- Debe conservar su empaque original.\n- Debe encontrarse en las mismas condiciones en las que fue recibido.\n- El cliente debe presentar la información relacionada con la compra.\n\nNo son aptos para devolución:\n\n- Productos perecederos como alimentos y flores.\n- Productos de higiene personal que hayan sido abiertos.\n- Productos que presenten señales de uso.\n- Productos que no conserven su empaque original.\n\nLas prendas de segunda mano están sujetas a revisión antes de aprobar una devolución.\n\nPara iniciar una devolución, el cliente debe ingresar a la sección de ayuda de EcoMarket y seguir las instrucciones i

## 3. Segmentación de documentos

Para dividir los documentos en fragmentos manejables se utiliza
`RecursiveCharacterTextSplitter`.

Se emplean inicialmente los siguientes parámetros:

- `chunk_size = 500`
- `chunk_overlap = 80`

El objetivo es conservar suficiente contexto en cada fragmento y reducir la
pérdida de información cuando una idea queda cerca del límite entre dos chunks.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documentos)

print(f"Documentos originales: {len(documentos)}")
print(f"Chunks generados: {len(chunks)}")

Documentos originales: 19
Chunks generados: 22


In [8]:
print("Contenido del primer chunk:")
print(chunks[0].page_content)

print("\nMetadatos:")
print(chunks[0].metadata)

Contenido del primer chunk:
# Política de Devoluciones de EcoMarket

EcoMarket permite realizar devoluciones dentro de los 30 días posteriores a la fecha de compra.

Para que un producto sea apto para devolución debe cumplir las siguientes condiciones:

- El producto debe estar sin usar.
- Debe conservar su empaque original.
- Debe encontrarse en las mismas condiciones en las que fue recibido.
- El cliente debe presentar la información relacionada con la compra.

No son aptos para devolución:

Metadatos:
{'fuente': 'politica_devoluciones.md', 'tipo': 'politica_devoluciones'}


## 4. Generación de embeddings

En esta etapa se utiliza el modelo:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Este modelo convierte cada chunk de texto en una representación vectorial.

Los embeddings permiten comparar semánticamente la consulta del usuario con
los fragmentos almacenados en la base de conocimiento.

In [9]:
modelo_embeddings = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=modelo_embeddings
)

print("Modelo de embeddings cargado correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado correctamente.


In [10]:
vector_prueba = embeddings.embed_query(
    "¿Puedo devolver un producto de EcoMarket?"
)

print("Número de dimensiones del embedding:", len(vector_prueba))
print("Primeros 10 valores:")
print(vector_prueba[:10])

Número de dimensiones del embedding: 384
Primeros 10 valores:
[0.25049689412117004, 0.024103866890072823, 0.03152767941355705, -0.12980912625789642, 0.3166915476322174, -0.0882209911942482, 0.013202176429331303, -0.19559195637702942, -0.1131434440612793, -0.05179305374622345]


## 5. Creación de la base de datos vectorial

En esta etapa, los embeddings generados a partir de los chunks se almacenan
en ChromaDB.

La base de datos vectorial permitirá recuperar los fragmentos más similares
a la consulta realizada por el usuario.

In [11]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ecomarket"
)

print("Base de datos vectorial creada correctamente.")

Base de datos vectorial creada correctamente.


## 6. Prueba del mecanismo de recuperación

Antes de conectar el modelo generativo, se verifica que ChromaDB recupere
fragmentos relevantes de la base de conocimiento.

In [12]:
consulta = "¿Puedo devolver un alimento?"

resultados = vectorstore.similarity_search(
    consulta,
    k=3
)

for i, resultado in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} ---")
    print("Fuente:", resultado.metadata.get("fuente"))
    print(resultado.page_content)


--- Resultado 1 ---
Fuente: productos.csv
Código del producto: P010
Nombre del producto: Frutas orgánicas surtidas
Categoría: Alimentos
Disponibilidad: Disponible
Admite devolución: No
Condición de devolución: Producto perecedero

--- Resultado 2 ---
Fuente: productos.csv
Código del producto: P007
Nombre del producto: Café orgánico
Categoría: Alimentos
Disponibilidad: Disponible
Admite devolución: No
Condición de devolución: Producto perecedero

--- Resultado 3 ---
Fuente: politica_devoluciones.md
No son aptos para devolución:

- Productos perecederos como alimentos y flores.
- Productos de higiene personal que hayan sido abiertos.
- Productos que presenten señales de uso.
- Productos que no conserven su empaque original.

Las prendas de segunda mano están sujetas a revisión antes de aprobar una devolución.

Para iniciar una devolución, el cliente debe ingresar a la sección de ayuda de EcoMarket y seguir las instrucciones indicadas.


## 7. Configuración del Retriever

El retriever utiliza la base vectorial para seleccionar los fragmentos
más relevantes para cada consulta.

Se configurará inicialmente para recuperar los tres fragmentos con mayor
similitud semántica.

In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever configurado correctamente.")

Retriever configurado correctamente.


## 8. Modelo de lenguaje

Se utiliza un modelo instructivo de propósito general para generar la respuesta
final a partir del contexto recuperado desde ChromaDB.

Para el prototipo se utiliza:

`Qwen/Qwen2.5-0.5B-Instruct`

Se selecciona un modelo pequeño para reducir los requerimientos computacionales
del ejercicio y permitir su ejecución en Google Colab.

In [14]:
!pip install -q langchain-huggingface

In [15]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

print("Importaciones realizadas correctamente.")

Importaciones realizadas correctamente.


In [30]:
nombre_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

modelo = AutoModelForCausalLM.from_pretrained(
    nombre_modelo,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelo Qwen 1.5B cargado correctamente.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo Qwen 1.5B cargado correctamente.


## 9. Prompt del sistema RAG

El modelo debe responder únicamente con base en la información recuperada
desde la base de conocimiento de EcoMarket.

Si el contexto no contiene información suficiente, el sistema debe indicarlo
claramente y evitar inventar información.

In [20]:
from langchain_core.prompts import PromptTemplate

plantilla_prompt = """
Eres un asistente virtual de atención al cliente de EcoMarket.

Responde siempre en español, de forma breve, clara, amable y profesional.

Debes utilizar exclusivamente la información proporcionada en el contexto.

Instrucciones:

1. Responde directamente la pregunta del cliente.
2. No repitas innecesariamente la misma información.
3. No inventes políticas, precios, productos, fechas ni condiciones.
4. Si la pregunta es general, responde utilizando las reglas generales disponibles
   en el contexto.
5. No menciones códigos de productos específicos a menos que el cliente pregunte
   por ese producto.
6. Si el contexto no contiene información suficiente para responder, responde
   exactamente:

   "No dispongo de información suficiente en la base de conocimiento de EcoMarket
   para responder esta solicitud."

Contexto:
{context}

Pregunta del cliente:
{input}

Respuesta breve:
"""

prompt = PromptTemplate.from_template(plantilla_prompt)

print("Prompt RAG mejorado correctamente.")

Prompt RAG mejorado correctamente.


## 10. Construcción de la cadena RAG

La cadena RAG conecta el retriever con el modelo generativo.

Primero se recuperan los fragmentos más relevantes desde ChromaDB.
Después, esos fragmentos se incorporan al prompt como contexto y se envían
al modelo de lenguaje para generar la respuesta final.

In [22]:
from langchain_core.runnables import RunnableLambda

def recuperar_contexto(pregunta):
    """
    Recupera los documentos más relevantes desde ChromaDB
    y los convierte en un único contexto de texto.
    """

    documentos_recuperados = retriever.invoke(pregunta)

    contexto = "\n\n".join(
        doc.page_content for doc in documentos_recuperados
    )

    return contexto


def preparar_prompt(pregunta):
    """
    Recupera el contexto y construye el prompt final
    que será enviado al modelo generativo.
    """

    contexto = recuperar_contexto(pregunta)

    prompt_final = prompt.format(
        context=contexto,
        input=pregunta
    )

    return prompt_final


cadena_rag = RunnableLambda(preparar_prompt) | llm

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [23]:
pregunta = "¿Puedo devolver un alimento?"

respuesta = cadena_rag.invoke(pregunta)

print("Pregunta:")
print(pregunta)

print("\nRespuesta del sistema:")
print(respuesta)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pregunta:
¿Puedo devolver un alimento?

Respuesta del sistema:
"El código del producto es P010, que incluye frutas orgánicas surtidas. Este producto no admite devolución debido a su categoría de alimentos y está sujetado a revisión antes de la devolución." 

Nota: Esta respuesta es basada en los criterios de EcoMarket y no es una evaluación precisa o recomendación. Para obtener una opinión más precisa, se recomienda consultar con un asesor técnico o consultar la página web oficial de EcoMarket. 

Si necesitas más información o tienes alguna otra pregunta, no dudes en preguntar. ¡Estoy aquí para ayudarte! 🍎🌱✨
La respuesta es:

"El código del producto es P010, que incluye frutas orgánicas surtidas. Este producto no admite devolución debido a su categoría de alimentos y está sujetado a revisión antes de la devolución." 

Espero que esto te sea útil. Si tienes alguna otra pregunta o necesitas más información, no dudes en preguntar. ¡Estoy aquí para ayudarte! 🍎🌱✨



## 11. Mejora de la generación de respuestas

La primera versión del sistema recuperó correctamente información relacionada
con los alimentos, pero el modelo generativo mezcló información proveniente de
diferentes fragmentos.

Para mejorar el seguimiento de instrucciones se utiliza el formato conversacional
propio del modelo Instruct y se limita la generación de la respuesta.

In [39]:
def preparar_entrada(pregunta):
    documentos_recuperados = retriever.invoke(pregunta)

    bloques = []

    for i, doc in enumerate(documentos_recuperados, start=1):
        bloques.append(
            f"""
EVIDENCIA {i}
Fuente: {doc.metadata.get('fuente')}
Contenido:
{doc.page_content}
"""
        )

    contexto = "\n".join(bloques)

    return {
        "pregunta": pregunta,
        "contexto": contexto
    }

In [37]:
def generar_respuesta(datos):

    system_message = """
Eres un asistente de atención al cliente de EcoMarket.

Tu respuesta debe basarse EXCLUSIVAMENTE en la evidencia recuperada.

REGLAS OBLIGATORIAS:

1. Lee primero toda la evidencia.
2. Si la evidencia dice explícitamente "Admite devolución: No",
   no puedes afirmar que el producto admite devolución.
3. Si la evidencia indica que una categoría no es apta para devolución,
   debes respetar esa política.
4. No inventes excepciones.
5. No contradigas el contexto.
6. No solicites información adicional si la pregunta ya puede responderse.
7. Responde en español.
8. Da una respuesta breve, de máximo 3 frases.
9. Si no existe información suficiente, responde exactamente:
   "No dispongo de información suficiente en la base de conocimiento de EcoMarket para responder esta solicitud."
"""

    user_message = f"""
EVIDENCIA RECUPERADA:

{datos['contexto']}

PREGUNTA:

{datos['pregunta']}

Analiza la evidencia y responde únicamente de acuerdo con ella.
"""

    mensajes = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto,
        return_tensors="pt"
    ).to(modelo.device)

    with torch.no_grad():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    tokens_generados = salida[0][entradas["input_ids"].shape[1]:]

    respuesta = tokenizer.decode(
        tokens_generados,
        skip_special_tokens=True
    )

    return respuesta.strip()

In [40]:
cadena_rag_mejorada = (
    RunnableLambda(preparar_entrada)
    | RunnableLambda(generar_respuesta)
)

print("Cadena RAG mejorada creada correctamente.")

Cadena RAG mejorada creada correctamente.


In [41]:
pregunta = "¿Puedo devolver un alimento?"

respuesta = cadena_rag_mejorada.invoke(pregunta)

print("Pregunta:")
print(pregunta)

print("\nRespuesta del sistema:")
print(respuesta)

Pregunta:
¿Puedo devolver un alimento?

Respuesta del sistema:
No, los alimentos no son aptos para devolución según la política de EcoMarket.


In [42]:
pregunta = "¿Puedo devolver un alimento?"

documentos_recuperados = retriever.invoke(pregunta)

print("DOCUMENTOS RECUPERADOS:\n")

for i, doc in enumerate(documentos_recuperados, start=1):
    print(f"--- Documento {i} ---")
    print("Fuente:", doc.metadata.get("fuente"))
    print(doc.page_content)
    print()

DOCUMENTOS RECUPERADOS:

--- Documento 1 ---
Fuente: productos.csv
Código del producto: P010
Nombre del producto: Frutas orgánicas surtidas
Categoría: Alimentos
Disponibilidad: Disponible
Admite devolución: No
Condición de devolución: Producto perecedero

--- Documento 2 ---
Fuente: productos.csv
Código del producto: P007
Nombre del producto: Café orgánico
Categoría: Alimentos
Disponibilidad: Disponible
Admite devolución: No
Condición de devolución: Producto perecedero

--- Documento 3 ---
Fuente: politica_devoluciones.md
No son aptos para devolución:

- Productos perecederos como alimentos y flores.
- Productos de higiene personal que hayan sido abiertos.
- Productos que presenten señales de uso.
- Productos que no conserven su empaque original.

Las prendas de segunda mano están sujetas a revisión antes de aprobar una devolución.

Para iniciar una devolución, el cliente debe ingresar a la sección de ayuda de EcoMarket y seguir las instrucciones indicadas.



## 12. Pruebas del sistema RAG

Para evaluar el comportamiento del sistema se realizan consultas sobre diferentes
tipos de información almacenada en la base de conocimiento.

Las pruebas permiten verificar:

- recuperación de información relevante;
- uso correcto del contexto;
- comportamiento ante preguntas sobre productos;
- comportamiento ante preguntas sobre envíos;
- respuesta cuando la información no está disponible.

In [43]:
def probar_consulta(pregunta):
    respuesta = cadena_rag_mejorada.invoke(pregunta)

    print("=" * 70)
    print("Pregunta:")
    print(pregunta)

    print("\nRespuesta:")
    print(respuesta)

In [44]:
probar_consulta(
    "¿Puedo devolver una botella reutilizable de acero inoxidable?"
)

Pregunta:
¿Puedo devolver una botella reutilizable de acero inoxidable?

Respuesta:
Sí, puedes devolver una botella reutilizable de acero inoxidable. El producto está disponible y admitido para devolución según la política de EcoMarket.


In [45]:
probar_consulta(
    "¿Qué hago si mi pedido está retrasado?"
)

Pregunta:
¿Qué hago si mi pedido está retrasado?

Respuesta:
El cliente debe verificar el estado del pedido en el enlace de seguimiento. Si el pedido se encuentra como retrasado, debe revisar el enlace de seguimiento para confirmarlo. Si aún así el pedido permanece retrasado después de realizar estas acciones, puede contactar al equipo de atención de EcoMarket para obtener más ayuda.


In [46]:
probar_consulta(
    "¿Puedo devolver un cepillo de dientes de bambú?"
)

Pregunta:
¿Puedo devolver un cepillo de dientes de bambú?

Respuesta:
El cepillo de dientes de bambú P002 está clasificado como producto de higiene personal y no admite devolución según la política de EcoMarket.


In [47]:
probar_consulta(
    "¿EcoMarket tiene tiendas físicas en Bogotá?"
)

Pregunta:
¿EcoMarket tiene tiendas físicas en Bogotá?

Respuesta:
No dispongo de información suficiente en la base de conocimiento de EcoMarket para responder esta solicitud.


In [48]:
probar_consulta(
    "¿Qué medios de pago acepta EcoMarket?"
)

Pregunta:
¿Qué medios de pago acepta EcoMarket?

Respuesta:
EcoMarket acepta tarjetas débito, tarjetas crédito y pagos electrónicos habilitados en la plataforma.


## 13. Resultados, limitaciones y supuestos

### Resultados

Las pruebas realizadas muestran que el sistema RAG es capaz de recuperar
información relevante desde la base de conocimiento de EcoMarket y utilizarla
para generar respuestas relacionadas con productos, devoluciones, envíos y
preguntas frecuentes.

También se verificó el comportamiento del sistema ante una consulta cuya
respuesta no estaba disponible en la base de conocimiento. En este caso, el
modelo indicó correctamente que no disponía de información suficiente, evitando
generar una respuesta sin respaldo.

Durante las pruebas iniciales se observó que un modelo generativo pequeño podía
interpretar de manera incorrecta algunos fragmentos recuperados, incluso cuando
el mecanismo de retrieval había identificado correctamente la información.

Para mejorar el comportamiento se realizaron ajustes en el prompt, haciendo más
explícitas las reglas que obligan al modelo a utilizar exclusivamente la evidencia
recuperada y a evitar contradicciones con el contexto.

### Limitaciones

La implementación corresponde a un prototipo académico y presenta algunas
limitaciones:

- La base de conocimiento contiene un número reducido de documentos.
- Los datos utilizados son ejemplos simulados de EcoMarket.
- La calidad de las respuestas depende tanto de la recuperación de información
  como de la capacidad del modelo generativo.
- Los parámetros de chunking fueron definidos inicialmente de forma experimental.
- No se implementó re-ranking de los documentos recuperados.
- No se conectaron bases de datos empresariales o APIs en tiempo real.
- El sistema utiliza un modelo de lenguaje de tamaño reducido para facilitar su
  ejecución en Google Colab.

### Supuestos

Para el desarrollo del prototipo se asumió que:

- Los documentos incluidos en la base de conocimiento contienen información
  válida y actualizada.
- Las consultas de los clientes se realizan principalmente en español.
- La cantidad de documentos es suficientemente pequeña para trabajar con
  ChromaDB en un entorno local.
- Los documentos recuperados desde la base vectorial son la única fuente de
  información permitida para generar las respuestas.

### Conclusiones

La implementación demuestra cómo una arquitectura RAG puede complementar un
modelo de lenguaje con información específica de una empresa.

El sistema desarrollado para EcoMarket recupera información mediante embeddings
y ChromaDB, incorpora los fragmentos relevantes al contexto y utiliza un modelo
generativo para producir la respuesta final.

Las pruebas también muestran que un buen mecanismo de recuperación no garantiza
por sí solo una respuesta correcta. La calidad final depende de la capacidad del
modelo generativo y de la forma en que se estructuran las instrucciones del prompt.